In [1]:
import pandas as pd
import numpy as np
import os
import json
import requests

from pyjstat import pyjstat
from collections import OrderedDict

In [2]:
dir_out = "../parsed_data/"

In [3]:
map_country_ISO = {
    "Belgium" : "BE",
    "Bulgaria" : "BG",
    "Czechia" : "CZ",
    "Denmark" : "DK",
    "Germany" : "DE",  ## Leon 
    "Estonia" : "EE",
    "Ireland" : "IE",
    "Greece" : "EL",   ## Leon: Greece = EL 
    "Spain" : "ES",
    "France" : "FR",
    "Croatia" : "HR",
    "Italy" : "IT",
    "Cyprus" : "CY",
    "Latvia" : "LV",
    "Lithuania" : "LT",
    "Luxembourg" : "LU",
    "Hungary" : "HU",
    "Malta" : "MT",
    "Netherlands" : "NL",
    "Austria" : "AT",
    "Poland" : "PL",
    "Portugal" : "PT",
    "Romania" : "RO",
    "Slovenia" : "SI",
    "Slovakia" : "SK",
    "Finland" : "FI",
    "Sweden" : "SE",
    "United Kingdom" : "UK",  ## Leon: UK nicht GB
    "Iceland" : "IS",
    "Norway" : "NO",
    "Montenegro" : "ME",
    "North Macedonia" : "MK",
    "Albania" : "AL",
    "Serbia" : "RS",
    "Türkiye" : "TR",       ## Leon 
    "Bosnia and Herzegovina" : "BA",
    "Kosovo*" : "XK",     ## Leon 
    "Moldova" : "MD",
    "Ukraine" : "UA",
    "Georgia" : "GE",
    "Liechtenstein":"LI"  ## Leon, added 
}

print(list(map_country_ISO.values()))

['BE', 'BG', 'CZ', 'DK', 'DE', 'EE', 'IE', 'EL', 'ES', 'FR', 'HR', 'IT', 'CY', 'LV', 'LT', 'LU', 'HU', 'MT', 'NL', 'AT', 'PL', 'PT', 'RO', 'SI', 'SK', 'FI', 'SE', 'UK', 'IS', 'NO', 'ME', 'MK', 'AL', 'RS', 'TR', 'BA', 'XK', 'MD', 'UA', 'GE', 'LI']


In [4]:
# Eurostat queries based on query builder: 
# https://ec.europa.eu/eurostat/web/query-builder/tool (Leon updated)
indicator = 'nrg_cb_e'
dataformat = 'JSON'

params = dict(
    siec = {'E7000'},
    sinceTimePeriod = '2014',
    geo = {'AT', 'BE', 'BG', 'CY', 'CZ', 'DE', 'DK', 'EE', 'EL', 'ES', 'FI', 'FR', 'HR', 'HU', 'IE', 'IT', 'LT', 'LU', 'LV', 'MD', 'MK', 'MT', 'NL', 'NO', 'PL', 'PT', 'RO', 'RS', 'SE', 'SI', 'SK', 'TR', 'UA', 'UK', 'AL', 'BA', 'LI', 'IS', 'GE', 'ME', 'XK'},
    unit = 'GWH',
    nrg_bal = {'NEP','IMP','EXP','DL'}
)

print(", ".join(params["geo"]))

IS, PL, IT, SK, DE, GE, RO, ME, PT, AL, UA, MT, SE, CZ, SI, CY, EL, FR, MD, LT, EE, TR, FI, XK, MK, LU, BA, LI, NO, AT, BG, HU, RS, LV, BE, ES, IE, DK, NL, HR, UK


In [5]:
# Leon: Inkonsistenzen zwischen Datenabfrage und Mapping eliminieren - in beide Richtungen: 

print(set(params["geo"]) - set(map_country_ISO.values()))
#print(set(map_country_ISO.values()) - set(params["geo"]))

set()


In [6]:
url = 'https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data/'+indicator+'?format='+dataformat
r = requests.get(url=url, params=params)

print(r.url)
#nrg_cb_e = pd.DataFrame(pyjstat.from_json_stat(r.json(object_pairs_hook=OrderedDict))[0])

https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data/nrg_cb_e?format=JSON&siec=E7000&sinceTimePeriod=2014&geo=IS&geo=PL&geo=IT&geo=SK&geo=DE&geo=GE&geo=RO&geo=ME&geo=PT&geo=AL&geo=UA&geo=MT&geo=SE&geo=CZ&geo=SI&geo=CY&geo=EL&geo=FR&geo=MD&geo=LT&geo=EE&geo=TR&geo=FI&geo=XK&geo=MK&geo=LU&geo=BA&geo=LI&geo=NO&geo=AT&geo=BG&geo=HU&geo=RS&geo=LV&geo=BE&geo=ES&geo=IE&geo=DK&geo=NL&geo=HR&geo=UK&unit=GWH&nrg_bal=DL&nrg_bal=EXP&nrg_bal=NEP&nrg_bal=IMP


In [7]:
# Data Frame aus Rohdaten: 
nrg_cb_e = pd.DataFrame(pyjstat.from_json_stat(r.json(object_pairs_hook=OrderedDict))[0])
nrg_cb_e.head()

,Time frequency,Energy balance,Standard international energy product classification (SIEC),Unit of measure,Geopolitical entity (reporting),Time,value
0,Annual,Imports,Electricity,Gigawatt-hour,Belgium,2014,21791.0
1,Annual,Imports,Electricity,Gigawatt-hour,Belgium,2015,23714.0
2,Annual,Imports,Electricity,Gigawatt-hour,Belgium,2016,14648.0
3,Annual,Imports,Electricity,Gigawatt-hour,Belgium,2017,14189.4
4,Annual,Imports,Electricity,Gigawatt-hour,Belgium,2018,21635.9


In [8]:
# Welche Länder sind in dem Rohdaten-Data Frame: 

nrg_cb_e['Geopolitical entity (reporting)'].unique()

array(['Belgium', 'Bulgaria', 'Czechia', 'Denmark', 'Germany', 'Estonia',
       'Ireland', 'Greece', 'Spain', 'France', 'Croatia', 'Italy',
       'Cyprus', 'Latvia', 'Lithuania', 'Luxembourg', 'Hungary', 'Malta',
       'Netherlands', 'Austria', 'Poland', 'Portugal', 'Romania',
       'Slovenia', 'Slovakia', 'Finland', 'Sweden', 'Iceland',
       'Liechtenstein', 'Norway', 'United Kingdom',
       'Bosnia and Herzegovina', 'Montenegro', 'Moldova',
       'North Macedonia', 'Georgia', 'Albania', 'Serbia', 'Türkiye',
       'Ukraine', 'Kosovo*'], dtype=object)

In [9]:
#rename countries and convert to MWh
nrg_cb_e = nrg_cb_e.dropna()  ## leon: kein Land durch dropna() verloren 
nrg_cb_e = nrg_cb_e.rename(columns={'Geopolitical entity (reporting)':'country'})
nrg_cb_e['country'] = nrg_cb_e['country'].map(map_country_ISO)
nrg_cb_e['load_MWh'] = nrg_cb_e['value'] * 1000
nrg_cb_e.head()

#Leon: auf Inkonsistenzen prüfen: 
#nrg_cb_e['country'].unique()
#print(set(nrg_cb_e['country']) - set(map_country_ISO.values()))
#print(set(map_country_ISO.values()) - set(nrg_cb_e['country']))
#print(set(nrg_cb_e['country']) - set(params["geo"]))
#print(set(params["geo"]) - set(nrg_cb_e['country']))

,Time frequency,Energy balance,Standard international energy product classification (SIEC),Unit of measure,country,Time,value,load_MWh
0,Annual,Imports,Electricity,Gigawatt-hour,BE,2014,21791.0,21791000.0
1,Annual,Imports,Electricity,Gigawatt-hour,BE,2015,23714.0,23714000.0
2,Annual,Imports,Electricity,Gigawatt-hour,BE,2016,14648.0,14648000.0
3,Annual,Imports,Electricity,Gigawatt-hour,BE,2017,14189.4,14189400.0
4,Annual,Imports,Electricity,Gigawatt-hour,BE,2018,21635.9,21635900.0


In [10]:
nrg_cb_e_pivot = nrg_cb_e.pivot_table(index=['country','Time'],columns='Energy balance',values='load_MWh')
nrg_cb_e_pivot['load_MWh'] = nrg_cb_e_pivot['Net electricity production'] + nrg_cb_e_pivot['Imports'] - nrg_cb_e_pivot['Exports'] + nrg_cb_e_pivot['Distribution losses']
nrg_cb_e_pivot['DL_share'] = nrg_cb_e_pivot['Distribution losses']/nrg_cb_e_pivot['load_MWh']
nrg_cb_e_pivot.head()

Energy balance  Distribution losses    Exports    Imports  \
country Time                                                
AL      2014                    NaN   183000.0  3250000.0   
        2015                    NaN   956000.0  2355000.0   
        2016                    NaN    42000.0        0.0   
        2017                    NaN        0.0  2914628.0   
        2018                    NaN  1763786.0   850480.0   

Energy balance  Net electricity production  load_MWh  DL_share  
country Time                                                    
AL      2014                     4708000.0       NaN       NaN  
        2015                     5866000.0       NaN       NaN  
        2016                     7720000.0       NaN       NaN  
        2017                     4497807.0       NaN       NaN  
        2018                     8508253.0       NaN       NaN

In [11]:
df_load = pd.DataFrame(nrg_cb_e_pivot['load_MWh'])

In [12]:
df_load = pd.pivot_table(df_load, values='load_MWh', index=['country'], columns=['Time'])
df_load.head()

Time,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
country,,,,,,,,,,,
AL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8759579.0,8852260.0,9631454.0
AT,NaN,NaN,NaN,74639510.0,74330516.0,74229713.0,71788986.0,75643154.0,78045607.0,74885762.0,76338645.0
BA,14521000.0,14712000.0,14210000.0,14745000.0,14655000.0,13911000.0,12813000.0,13959000.0,13121000.0,12779000.0,NaN
BE,87420900.0,88375400.0,88582100.0,89147100.0,89398300.0,88085000.0,85789600.0,88767900.0,87322700.0,85138000.0,86868927.0
BG,33781000.0,34151000.0,34851000.0,35871136.0,34907104.0,34510765.0,33814694.0,34679496.0,35399233.0,35169747.0,36539091.0


In [13]:
df_load.to_csv(dir_out+'load_yearly_Eurostat.csv', encoding="utf-8")

OSError: Cannot save file into a non-existent directory: '..\parsed_data'

In [14]:
nrg_cb_e_pivot['DL_share']

country  Time
AL       2014         NaN
         2015         NaN
         2016         NaN
         2017         NaN
         2018         NaN
                   ...   
XK       2020    0.139254
         2021    0.129712
         2022    0.000000
         2023    0.060146
         2024    0.000000
Name: DL_share, Length: 438, dtype: float64